### Importing required libraries

In [2]:
import cv2
import pandas as pd
from ultralytics import YOLO
import time
import csv

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


### Data logging

In [28]:
log_file = open("vehicle_data.csv","w",newline="")
writer = csv.writer(log_file)

writer.writerow(["timestamp","vehicle_id","vehicle_type","direction","speed_kmh"])

55

### Importing video

In [29]:
cap = cv2.VideoCapture("/content/cars_on_highways.mp4")

model = YOLO("yolov8s.pt")

fps = cap.get(cv2.CAP_PROP_FPS)
max_seconds = 30
max_frames = int(fps * max_seconds)

frame_count = 0

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter("output_new.mp4", fourcc, fps, (1020,500))

class_list = model.names

# tracking dictionaries
down = {}
up = {}

# time counters
start_up = {}
start_down = {}

speed_down = {}
speed_up = {}

# class-wise counters
down_count = {"car":set(), "bus":set(), "truck":set()}
up_count = {"car":set(), "bus":set(), "truck":set()}

red_line_y = 198
blue_line_y = 268
offset = 7

### Processing video

In [ ]:


while True:

    ret, frame = cap.read()
    if not ret:
        break

    frame_count += 1

    if frame_count > max_frames:
        break

    frame = cv2.resize(frame,(1020,500))

    results = model.track(frame, persist=True)

    if results[0].boxes.id is not None:

        boxes = results[0].boxes.xyxy.cpu().numpy()
        ids = results[0].boxes.id.cpu().numpy()
        classes = results[0].boxes.cls.cpu().numpy()

        for box, track_id, cls in zip(boxes, ids, classes):

            x1,y1,x2,y2 = map(int, box)
            id = int(track_id)

            label = class_list[int(cls)]

            if label in ["car","bus","truck"]:

                cx = int((x1+x2)/2)
                cy = int((y1+y2)/2)

                cv2.rectangle(frame,(x1,y1),(x2,y2),(0,255,0),2)
                cv2.circle(frame,(cx,cy),4,(255,0,255),-1)

                cv2.putText(frame,label+"-"+str(id),(x1,y1-10),
                            cv2.FONT_HERSHEY_SIMPLEX,0.5,(255,255,255),1)

                # DOWN Direction

                if red_line_y < (cy+offset) and red_line_y > (cy-offset):
                    down[id] = label
                    if id not in start_down:
                        start_down[id] = time.time()

                if id in down:
                    if blue_line_y < (cy+offset) and blue_line_y > (cy-offset):

                        vehicle_class = down[id]

                        # calculate speed only once
                        if id not in speed_down and id in start_down:

                            elapsed_time_down = time.time() - start_down[id]

                            distance = 10
                            speed_ms_down = distance / elapsed_time_down
                            speed_kmh_down = speed_ms_down * 3.6

                            speed_down[id] = int(speed_kmh_down)

                            writer.writerow([
                                      time.time(),
                                      id,
                                      vehicle_class,
                                      "down",
                                      speed_down[id]
                                  ])


                            if id not in down_count[vehicle_class]:
                                down_count[vehicle_class].add(id)


                        # display stored speed
                        if id in speed_down:
                            cv2.putText(frame,
                                        str(speed_down[id])+" km/h",
                                        (x1,y1-10),
                                        cv2.FONT_HERSHEY_SIMPLEX,
                                        0.8,
                                        (0,255,255),
                                        2)

                # UP direction
                if blue_line_y < (cy+offset) and blue_line_y > (cy-offset):
                    up[id] = label
                    if id not in start_up:
                        start_up[id] = time.time()

                if id in up:
                    if red_line_y < (cy+offset) and red_line_y > (cy-offset):

                        vehicle_class = up[id]

                        # calculate speed only once
                        if id not in speed_up and id in start_up:

                            elapsed_time_up = time.time() - start_up[id]

                            distance = 10
                            speed_ms_up = distance / elapsed_time_up
                            speed_kmh_up = speed_ms_up * 3.6

                            speed_up[id] = int(speed_kmh_up)

                            writer.writerow([
                                      time.time(),
                                      id,
                                      vehicle_class,
                                      "up",
                                      speed_up[id]
                                  ])

                            if id not in up_count[vehicle_class]:
                                up_count[vehicle_class].add(id)

                        # display stored speed
                        if id in speed_up:
                            cv2.putText(frame,
                                        str(speed_up[id])+" km/h",
                                        (x1,y1-10),
                                        cv2.FONT_HERSHEY_SIMPLEX,
                                        0.8,
                                        (0,255,255),
                                        2)

    # draw lines
    cv2.line(frame,(172,198),(774,198),(0,0,255),3)
    cv2.line(frame,(8,268),(927,268),(255,0,0),3)

    # DOWN counts (top-left)
    cv2.putText(frame,"DOWN",(30,40),
                cv2.FONT_HERSHEY_SIMPLEX,0.8,(0,255,0),2)

    cv2.putText(frame,"Cars: "+str(len(down_count["car"])),(30,70),
                cv2.FONT_HERSHEY_SIMPLEX,0.7,(255,255,255),2)

    cv2.putText(frame,"Bus: "+str(len(down_count["bus"])),(30,100),
                cv2.FONT_HERSHEY_SIMPLEX,0.7,(255,255,255),2)

    cv2.putText(frame,"Truck: "+str(len(down_count["truck"])),(30,130),
                cv2.FONT_HERSHEY_SIMPLEX,0.7,(255,255,255),2)

    # UP counts (top-right)
    cv2.putText(frame,"UP",(820,40),
                cv2.FONT_HERSHEY_SIMPLEX,0.8,(0,255,255),2)

    cv2.putText(frame,"Cars: "+str(len(up_count["car"])),(820,70),
                cv2.FONT_HERSHEY_SIMPLEX,0.7,(255,255,255),2)

    cv2.putText(frame,"Bus: "+str(len(up_count["bus"])),(820,100),
                cv2.FONT_HERSHEY_SIMPLEX,0.7,(255,255,255),2)

    cv2.putText(frame,"Truck: "+str(len(up_count["truck"])),(820,130),
                cv2.FONT_HERSHEY_SIMPLEX,0.7,(255,255,255),2)

    out.write(frame)

cap.release()
out.release()
# cv2.destroyAllWindows()
print('Output video is ready.')

In [31]:
avg_speed_down = int(sum(speed_down.values())/len(speed_down)) if speed_down else 0
avg_speed_up = int(sum(speed_up.values())/len(speed_up)) if speed_up else 0

In [33]:
df = pd.read_csv('/content/vehicle_data.csv')
df.shape

(69, 5)

In [34]:
vehicle_counts = df["vehicle_type"].value_counts()
vehicle_counts

,count
vehicle_type,
car,55
truck,14
